In [8]:
### Run these pip installs if imports are not available
# !pip install plotly
# !pip install folium


# Core imports
import math
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Optional visualization imports
import plotly.graph_objects as go
import folium

from ipywidgets import Layout, HBox, VBox, Tab, Accordion

In [10]:
# Sample Property Data

property_data = {
    "address": "Fort Worth, Texas 76244",
    "project_location": "McPherson Ranch Neighborhood",
    "lot_size_acres": 0.1263,
    "lot_size_sqft": 5500,
    "zoning": "A-5",
    "permitted_use": "Single family residence",
    "neighborhood_description": "New (post 2000) residential community; high opportunity area",
    "existing_home_sqft": 1524,
    "crime_incidents_500ft": 2,
    "median_household_income": 122217,
    "fwclt_suitability_score": 302,
    "fwclt_suitability_total": 415,
    "cost_burden_score": 4,
    "moderately_priced_homes_score": 4,
    "housing_transportation_score": 4,
    "displacement_score": 1,
    "child_opportunity_score": 8,
    "floodplain": "No",
    "brownfield": "No",
    "road_condition": "Good",
    "street_lighting_score": 10,
    "sidewalk_score": 10,
    "transit": "No nearby bus stops",
    "public_water": "Yes",
    "sanitary": "Yes",
    "environmental_listings": "TBD",
    "latitude": 32.96,    # placeholder - replace with real coordinates
    "longitude": -97.29   # placeholder - replace with real coordinates
}

amenities = [
    {"name": "Kroger", "type": "Grocery", "distance_miles": 1.2, "lat": 32.965, "lon": -97.285},
    {"name": "Walgreens", "type": "Pharmacy", "distance_miles": 1.3, "lat": 32.962, "lon": -97.281},
    {"name": "Integrity Urgent Care", "type": "Healthcare", "distance_miles": 1.5, "lat": 32.958, "lon": -97.278},
    {"name": "Kay Granger Elementary School", "type": "School", "distance_miles": 1.3, "lat": 32.969, "lon": -97.292},
    {"name": "McPherson Park", "type": "Park", "distance_miles": 0.15, "lat": 32.961, "lon": -97.287},
    {"name": "Golden Triangle Library", "type": "Library", "distance_miles": 2.0, "lat": 32.967, "lon": -97.276},
]

amenities_df = pd.DataFrame(amenities)
amenities_df

,name,type,distance_miles,lat,lon
0,Kroger,Grocery,1.20,32.965,-97.285
1,Walgreens,Pharmacy,1.30,32.962,-97.281
2,Integrity Urgent Care,Healthcare,1.50,32.958,-97.278
3,Kay Granger Elementary School,School,1.30,32.969,-97.292
4,McPherson Park,Park,0.15,32.961,-97.287
5,Golden Triangle Library,Library,2.00,32.967,-97.276


In [11]:
# Scorecard Structure
scorecard_rows = [
    ("Neighborhood and special designations", 0.05, 16),
    ("Nearby community amenities", 0.05, 17),
    ("City services", 0.05, 19),
    ("Vacant lots / dilapidated housing", 0.05, 19),
    ("Safety, lighting, and public capital improvements", 0.05, 19),
    ("Land use and zoning classifications", 0.15, 19),
    ("Construction and development cost impacts", 0.05, 15),
    ("Development plans", 0.05, 15),
    ("Financial projections and acquisition pricing", 0.20, 15),
    ("Neighborhood classifications / sentiment", 0.05, 19),
    ("Displacement risk / comparable land values", 0.05, 20),
    ("Neighborhood strategy measures", 0.05, 17),
    ("Community and governmental factors", 0.05, 18),
    ("Investment risk assessment", 0.10, 16),
]

score_df = pd.DataFrame(scorecard_rows, columns=["Factor", "Weight", "Initial Score"])
score_df["Weighted Score"] = score_df["Weight"] * score_df["Initial Score"]
score_df

,Factor,Weight,Initial Score,Weighted Score
0,Neighborhood and special designations,0.05,16,0.80
1,Nearby community amenities,0.05,17,0.85
2,City services,0.05,19,0.95
3,Vacant lots / dilapidated housing,0.05,19,0.95
4,"Safety, lighting, and public capital improvements",0.05,19,0.95
5,Land use and zoning classifications,0.15,19,2.85
6,Construction and development cost impacts,0.05,15,0.75
7,Development plans,0.05,15,0.75
8,Financial projections and acquisition pricing,0.20,15,3.00
9,Neighborhood classifications / sentiment,0.05,19,0.95


In [12]:
# Dashboard Styling
display(HTML('''
<style>
.dashboard-title {
    font-size: 30px;
    font-weight: 800;
    margin-bottom: 8px;
    color: #102a43;
}
.dashboard-subtitle {
    font-size: 15px;
    color: #486581;
    margin-bottom: 18px;
}
.metric-card {
    background: linear-gradient(135deg, #f8fbff, #eef6ff);
    border: 1px solid #d9e8f5;
    border-radius: 16px;
    padding: 16px 18px;
    margin: 8px 8px 8px 0;
    box-shadow: 0 6px 18px rgba(16,42,67,0.06);
    min-width: 210px;
}
.metric-label {
    font-size: 13px;
    color: #627d98;
    margin-bottom: 6px;
}
.metric-value {
    font-size: 24px;
    font-weight: 800;
    color: #102a43;
}
.section-card {
    background: white;
    border: 1px solid #e6edf3;
    border-radius: 16px;
    padding: 16px;
    margin-top: 10px;
}
.good-pill, .warn-pill, .bad-pill {
    display: inline-block;
    padding: 6px 12px;
    border-radius: 999px;
    font-weight: 700;
    font-size: 12px;
}
.good-pill { background: #e3fcef; color: #0b6b3a; }
.warn-pill { background: #fff4db; color: #8a5a00; }
.bad-pill  { background: #fde8e8; color: #9b1c1c; }
</style>
'''))